# Firehose 1K — Complete IAA Analysis
**NeurIPS Paper · MPI SWS**

Pipeline:
1. Ines vs adash — before discussion
2. Ines vs adash — after discussion (21 resolved)
3. Tiebreaker — soumidas majority vote on 10 remaining
4. Final stats on all 1000 posts + Bluesky comparison

> **Note on Bluesky comparison**: Bluesky = predicted, Human = ground truth.
> We report **Unsafe-class** precision/recall only (the class of interest).
> Safe-class metrics are not meaningful here.

## 1. Setup

In [ ]:
import json, warnings
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.metrics import cohen_kappa_score, confusion_matrix, classification_report
warnings.filterwarnings("ignore")

ANN_PATH      = "Bluesky_Moderation_Annotations_-_Sheet1.csv"
DISAGREE_PATH = "firehose_disagreements.csv"
POSTS_FH      = "/NS/bluesky/nobackup/ines/neurips/hf_upload/data/firehose_1k_labeled.jsonl"


## 2. Load Data

In [ ]:
ann = pd.read_csv(ANN_PATH)
ann.columns = ann.columns.str.strip()
ann = ann.drop_duplicates(subset=['annotator_name','post_id'], keep='last')

# Firehose: Ines and adash only (display_num 10000–20000)
fh = ann[
    (ann['display_num'] >= 10000) &
    (ann['display_num'] <  20000) &
    (ann['annotator_name'].isin(['Ines','adash']))
].copy()

# Tiebreaker: soumidas (display_num 30000–40000)
tb = ann[
    (ann['display_num'] >= 30000) &
    (ann['display_num'] <  40000) &
    (ann['annotator_name'] == 'soumidas')
].copy()

print("Firehose annotations:")
print(fh.groupby('annotator_name').size().to_string())
print(f"\nTiebreaker (soumidas): {len(tb)} posts")

# Load firehose posts (for Bluesky labels)
fh_posts = {}
try:
    with open(POSTS_FH) as f:
        for line in f:
            if line.strip():
                p = json.loads(line)
                if p.get('uri'):
                    fh_posts[p['uri']] = p
    print(f"\nFirehose posts loaded: {len(fh_posts):,}")
    n_bsky = sum(1 for p in fh_posts.values() if p.get('bluesky_labels'))
    print(f"Bluesky-flagged in dataset: {n_bsky}")
except FileNotFoundError:
    print("\nWARNING: Posts file not found — run on server for Bluesky analysis")

def build_lookup(df, name):
    sub = df[df['annotator_name']==name].drop_duplicates('post_id', keep='last')
    return dict(zip(sub['post_id'], sub['label']))

lu_ines  = build_lookup(fh, 'Ines')
lu_adash = build_lookup(fh, 'adash')
lu_tb    = build_lookup(tb, 'soumidas')

common = sorted(set(lu_ines) & set(lu_adash))
print(f"\nInes+adash overlap: {len(common)} posts")


## 3. Label Distribution

In [ ]:
dist = fh.groupby(['annotator_name','label']).size().unstack(fill_value=0)
dist['total']    = dist.sum(axis=1)
dist['% unsafe'] = (dist.get('Unsafe',0)/dist['total']*100).round(1)
print(dist)

if fh_posts:
    bsky_flagged = sum(1 for p in common if fh_posts.get(p,{}).get('bluesky_labels'))
    print(f"\nBluesky flagged: {bsky_flagged}/{len(common)} ({100*bsky_flagged/len(common):.1f}%)")


## 4. Helper Functions

In [ ]:
cats = ['Safe', 'Unsafe']

def iaa_stats(common_list, lu_a, lu_b, name_a='Ines', name_b='adash'):
    la = [lu_a[p] for p in common_list]
    lb = [lu_b[p] for p in common_list]
    agree = sum(x==y for x,y in zip(la,lb))
    pct   = 100*agree/len(common_list)
    kappa = cohen_kappa_score(la, lb)
    interp = ('almost perfect' if kappa>=0.8 else 'substantial' if kappa>=0.6
              else 'moderate' if kappa>=0.4 else 'fair' if kappa>=0.2 else 'poor')
    print(f"n={len(common_list)}  agree={agree} ({pct:.1f}%)  κ={kappa:.3f} ({interp})")
    return la, lb, kappa

def bsky_stats(common_list, label_map, annotator_name='Human'):
    """
    Bluesky (predicted) vs Human (ground truth) — Unsafe class only.
    label_map: dict {uri -> 'Safe'|'Unsafe'}
    """
    if not fh_posts:
        print("Bluesky stats not available (posts file not loaded)")
        return None
    y_true = [1 if label_map[p]=='Unsafe' else 0 for p in common_list]
    y_pred = [1 if fh_posts.get(p,{}).get('bluesky_labels') else 0 for p in common_list]

    tp = sum(1 for t,p in zip(y_true,y_pred) if t==1 and p==1)
    fp = sum(1 for t,p in zip(y_true,y_pred) if t==0 and p==1)
    fn = sum(1 for t,p in zip(y_true,y_pred) if t==1 and p==0)
    tn = sum(1 for t,p in zip(y_true,y_pred) if t==0 and p==0)

    prec   = tp/(tp+fp) if (tp+fp)>0 else 0
    recall = tp/(tp+fn) if (tp+fn)>0 else 0
    f1     = 2*prec*recall/(prec+recall) if (prec+recall)>0 else 0

    n_bsky        = sum(y_pred)
    n_human_unsafe= sum(y_true)

    print(f"  Bluesky flagged ({annotator_name} posts):  {n_bsky} ({100*n_bsky/len(y_pred):.1f}%)")
    print(f"  {annotator_name} unsafe:                   {n_human_unsafe} ({100*n_human_unsafe/len(y_true):.1f}%)")
    print(f"  TP (both unsafe):      {tp}")
    print(f"  FN (human only):       {fn}  ← Bluesky missed")
    print(f"  FP (Bluesky only):     {fp}  ← Bluesky false positive")
    print(f"  Precision (Unsafe): {prec:.3f}")
    print(f"  Recall    (Unsafe): {recall:.3f}")
    print(f"  F1        (Unsafe): {f1:.3f}")

    return pd.DataFrame([[tn,fp],[fn,tp]],
        index=['Human: Safe','Human: Unsafe'],
        columns=['Bsky: Safe','Bsky: Unsafe'])

def cm_plot(ax, cm_df, title, cmap):
    sns.heatmap(cm_df, annot=True, fmt='d', cmap=cmap, ax=ax,
                cbar=False, linewidths=0.5, linecolor='white')
    ax.set_title(title)


---
## Part 1 — Before Discussion

In [ ]:
print("=== BEFORE DISCUSSION ===\n")
la_b, lb_b, kappa_b = iaa_stats(common, lu_ines, lu_adash)
disagree_before = [p for p in common if lu_ines[p] != lu_adash[p]]
print(f"Disagreements: {len(disagree_before)}")


In [ ]:
print("\nClassification report (Ines vs adash, before discussion):")
print(classification_report(la_b, lb_b, target_names=['Safe','Unsafe']))


In [ ]:
print("\nBluesky vs Ines (before discussion):")
cm_bsky_ines_b = bsky_stats(common, lu_ines, annotator_name='Ines')

print("\nBluesky vs adash (before discussion):")
cm_bsky_adash_b = bsky_stats(common, lu_adash, annotator_name='adash')


In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(15,4))

# IAA confusion matrix
cm_b = confusion_matrix(la_b, lb_b, labels=cats)
cm_plot(axes[0],
    pd.DataFrame(cm_b,
        index=['Ines: Safe','Ines: Unsafe'],
        columns=['adash: Safe','adash: Unsafe']),
    f'IAA Before Discussion\nn={len(common)}  κ={kappa_b:.3f}',
    'Blues')

# Bluesky vs Ines
if fh_posts and cm_bsky_ines_b is not None:
    cm_plot(axes[1], cm_bsky_ines_b,
        'Bluesky vs Ines (Before Discussion)',
        'Oranges')
else:
    axes[1].axis('off')

# Bluesky vs adash
if fh_posts and cm_bsky_adash_b is not None:
    cm_plot(axes[2], cm_bsky_adash_b,
        'Bluesky vs adash (Before Discussion)',
        'Reds')
else:
    axes[2].axis('off')

plt.tight_layout()
plt.savefig('fh_before_discussion.png', dpi=150, bbox_inches='tight')
plt.show()


---
## Part 2 — After Discussion

In [ ]:
disagree_df = pd.read_csv(DISAGREE_PATH)
disagree_df['Ines_label']  = disagree_df['Ines_label'].str.strip().str.capitalize()
disagree_df['adash_label'] = disagree_df['adash_label'].str.strip().str.capitalize()

resolved     = disagree_df[disagree_df['Ines_label'] == disagree_df['adash_label']]
still        = disagree_df[disagree_df['Ines_label'] != disagree_df['adash_label']]
resolved_map = dict(zip(resolved['uri'], resolved['Ines_label']))
still_uris   = set(still['uri'])

print(f"Original disagreements: {len(disagree_df)}")
print(f"Resolved:               {len(resolved)}")
print(f"Still unresolved:       {len(still)}\n")
print("Resolved breakdown (what label was agreed on):")
print(resolved['Ines_label'].value_counts().to_string())


In [ ]:
# Override resolved posts with agreed label, keep original for unresolved
lu_ines_post  = {p: resolved_map.get(p, lu_ines[p])  for p in common}
lu_adash_post = {p: resolved_map.get(p, lu_adash[p]) for p in common}

print("=== AFTER DISCUSSION (n=1000, incl. 10 still unresolved) ===\n")
la_a, lb_a, kappa_a = iaa_stats(common, lu_ines_post, lu_adash_post)


In [ ]:
print("\nClassification report (after discussion):")
print(classification_report(la_a, lb_a, target_names=['Safe','Unsafe']))


In [ ]:
print("\nBluesky vs Ines (after discussion):")
cm_bsky_ines_a = bsky_stats(common, lu_ines_post, annotator_name='Ines')

print("\nBluesky vs adash (after discussion):")
cm_bsky_adash_a = bsky_stats(common, lu_adash_post, annotator_name='adash')


In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(15,4))

cm_a = confusion_matrix(la_a, lb_a, labels=cats)
cm_plot(axes[0],
    pd.DataFrame(cm_a,
        index=['Ines: Safe','Ines: Unsafe'],
        columns=['adash: Safe','adash: Unsafe']),
    f'IAA After Discussion\nn={len(common)}  κ={kappa_a:.3f}',
    'Greens')

if fh_posts and cm_bsky_ines_a is not None:
    cm_plot(axes[1], cm_bsky_ines_a,
        'Bluesky vs Ines (After Discussion)',
        'Oranges')
else:
    axes[1].axis('off')

if fh_posts and cm_bsky_adash_a is not None:
    cm_plot(axes[2], cm_bsky_adash_a,
        'Bluesky vs adash (After Discussion)',
        'Reds')
else:
    axes[2].axis('off')

plt.tight_layout()
plt.savefig('fh_after_discussion.png', dpi=150, bbox_inches='tight')
plt.show()


---
## Part 3 — Tiebreaker (soumidas on 10 unresolved posts)

In [ ]:
print(f"=== TIEBREAKER — soumidas on {len(still_uris)} posts ===\n")

final_tb = {}
tb_rows  = []
for uri in sorted(still_uris):
    i = lu_ines.get(uri, '?')
    a = lu_adash.get(uri, '?')
    s = lu_tb.get(uri, None)
    votes    = [i, a] + ([s] if s else [])
    majority = 'Unsafe' if votes.count('Unsafe') > votes.count('Safe') else 'Safe'
    final_tb[uri] = majority
    tb_rows.append({
        'Ines': i, 'adash': a,
        'soumidas': s if s else '—',
        'majority': majority,
        'text': str(fh_posts.get(uri,{}).get('text',''))[:80] if fh_posts else ''
    })

tb_df = pd.DataFrame(tb_rows)
print(tb_df[['Ines','adash','soumidas','majority','text']].to_string(index=False))
print(f"\nMajority → Unsafe: {(tb_df['majority']=='Unsafe').sum()}")
print(f"Majority → Safe:   {(tb_df['majority']=='Safe').sum()}")


---
## Part 4 — Final Stats (all 1000 posts)

In [ ]:
# Build final label map
# - Posts that were originally agreed: keep original label
# - Posts resolved in discussion: use agreed label
# - Posts resolved by tiebreaker: use majority vote
final_map = {}
for p in common:
    if p in final_tb:
        final_map[p] = final_tb[p]
    elif p in resolved_map:
        final_map[p] = resolved_map[p]
    else:
        final_map[p] = lu_ines[p]  # originally agreed → same for both

n_unsafe = sum(1 for v in final_map.values() if v=='Unsafe')
n_safe   = len(final_map) - n_unsafe

print(f"=== FINAL LABEL DISTRIBUTION (n={len(final_map)}) ===")
print(f"  Unsafe: {n_unsafe} ({100*n_unsafe/len(final_map):.1f}%)")
print(f"  Safe:   {n_safe}  ({100*n_safe/len(final_map):.1f}%)")
print(f"\nHow labels were determined:")
print(f"  Originally agreed (no discussion needed): {len(common)-len(disagree_df)}")
print(f"  Resolved through discussion:              {len(resolved)}")
print(f"  Resolved through tiebreaker:              {len(final_tb)}")


In [ ]:
# Each annotator vs final consensus
print("=== ANNOTATOR vs FINAL CONSENSUS ===\n")
for name, lu in [('Ines', lu_ines), ('adash', lu_adash)]:
    la_f = [lu[p]         for p in common]
    lb_f = [final_map[p]  for p in common]
    ag   = sum(x==y for x,y in zip(la_f, lb_f))
    k    = cohen_kappa_score(la_f, lb_f)
    print(f"{name} vs final consensus:")
    print(f"  Agreement: {ag}/{len(common)} ({100*ag/len(common):.1f}%)  κ={k:.3f}\n")


In [ ]:
print("=== BLUESKY vs FINAL CONSENSUS (Unsafe class only) ===\n")
cm_bsky_f = bsky_stats(common, final_map, annotator_name='Final consensus')
if cm_bsky_f is not None:
    print(f"\nConfusion matrix:")
    print(cm_bsky_f.to_string())


In [ ]:
# ── 3-panel progression plot ──────────────────────────────────────────────────
fig, axes = plt.subplots(1, 3, figsize=(15, 5))

# Before
cm_b = confusion_matrix(la_b, lb_b, labels=cats)
cm_plot(axes[0],
    pd.DataFrame(cm_b,
        index=['Ines: Safe','Ines: Unsafe'],
        columns=['adash: Safe','adash: Unsafe']),
    f'Before Discussion\nn={len(common)}  κ={kappa_b:.3f}',
    'Blues')

# After
cm_a = confusion_matrix(la_a, lb_a, labels=cats)
cm_plot(axes[1],
    pd.DataFrame(cm_a,
        index=['Ines: Safe','Ines: Unsafe'],
        columns=['adash: Safe','adash: Unsafe']),
    f'After Discussion\nn={len(common)}  κ={kappa_a:.3f}',
    'Greens')

# Ines vs final
la_f = [lu_ines[p]  for p in common]
lb_f = [final_map[p] for p in common]
k_f  = cohen_kappa_score(la_f, lb_f)
cm_f = confusion_matrix(la_f, lb_f, labels=cats)
cm_plot(axes[2],
    pd.DataFrame(cm_f,
        index=['Ines: Safe','Ines: Unsafe'],
        columns=['Final: Safe','Final: Unsafe']),
    f'Ines vs Final Consensus\nn={len(common)}  κ={k_f:.3f}',
    'Purples')

plt.suptitle('Firehose 1K — IAA Progression', fontsize=13, y=1.02)
plt.tight_layout()
plt.savefig('fh_iaa_progression.png', dpi=150, bbox_inches='tight')
plt.show()


In [ ]:
# ── Summary table ─────────────────────────────────────────────────────────────
summary = pd.DataFrame([
    {'Stage': 'Before discussion',  'n': len(common), 'Agreement (%)': round(100*sum(x==y for x,y in zip(la_b,lb_b))/len(common),1), "Cohen's κ": round(kappa_b,3)},
    {'Stage': 'After discussion',   'n': len(common), 'Agreement (%)': round(100*sum(x==y for x,y in zip(la_a,lb_a))/len(common),1), "Cohen's κ": round(kappa_a,3)},
    {'Stage': 'Ines vs consensus',  'n': len(common), 'Agreement (%)': round(100*sum(x==y for x,y in zip(la_f,lb_f))/len(common),1), "Cohen's κ": round(k_f,3)},
])
print(summary.to_string(index=False))


In [ ]:
# ── Save final dataset ────────────────────────────────────────────────────────
rows = []
for p in common:
    post = fh_posts.get(p, {}) if fh_posts else {}
    rows.append({
        'uri':           p,
        'text':          str(post.get('text',''))[:300],
        'final_label':   final_map[p],
        'Ines':          lu_ines[p],
        'adash':         lu_adash[p],
        'soumidas_tb':   lu_tb.get(p, ''),
        'bluesky_labels':','.join(post.get('bluesky_labels',[])),
        'how_determined':('tiebreaker' if p in final_tb
                          else 'discussion' if p in resolved_map
                          else 'original_agreement'),
    })
final_df = pd.DataFrame(rows)
final_df.to_csv('firehose_final_labels.csv', index=False)
print(f"Saved {len(final_df)} posts → firehose_final_labels.csv")
print(final_df['final_label'].value_counts().to_string())
print(f"\nHow determined:")
print(final_df['how_determined'].value_counts().to_string())
final_df.head()
